# SigLIP2 furniture text-strategy benchmark

Screen five ways to handle long furniture descriptions under controlled data, initialization, and optimizer settings:

1. **baseline64** — original text, right-truncated to 64 tokens;
2. **compact64** — category plus TF-IDF/visual-signal sentence selection within 64 tokens;
3. **multichunk64** — up to four overlapping chunks, rotated during training and mean-pooled for product text embeddings;
4. **dual64** — separate description and taxonomy losses, with separate embeddings fused at inference;
5. **extended128** — experimental 128-position text tower initialized by interpolating the pretrained 64-position table.

This is a bounded screening experiment. It does not overwrite previous checkpoints or datasets.

In [1]:
from contextlib import nullcontext
from pathlib import Path
import gc
import json
import platform
import random
import re
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.feature_extraction.text import TfidfVectorizer
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoProcessor

SEED = 42
MODEL_ID = "google/siglip2-base-patch16-224"
TRAIN_ROWS = 256
EVAL_ROWS = 128
CATALOG_ROWS = 500
EPOCHS = 2
TRAIN_BATCH_SIZE = 4
ENCODE_BATCH_SIZE = 8
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
DUAL_TAXONOMY_LOSS_WEIGHT = 0.30
DUAL_TAXONOMY_FUSION_WEIGHT = 0.20
MAX_CHUNKS = 4
CHUNK_OVERLAP = 12
QUERY_LEAF_COUNT = 20
TOP_K = 10
TEXT_QUERY = "mid-century modern walnut coffee table for a living room"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
PIN_MEMORY = DEVICE.type == "cuda"
print({"python": platform.python_version(), "device": str(DEVICE), "torch": torch.__version__})
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("Select the RecSystem — SigLIP2 (Python 3.11) kernel.")

{'python': '3.11.6', 'device': 'mps', 'torch': '2.2.2'}


## 1. Load the preserved 20k data and fixed splits

In [2]:
PAIRS_RELATIVE_PATH = Path("data/meta_Home_and_Kitchen_furniture_siglip2_pairs_20k_taxonomy_enriched.csv")
SPLIT_RELATIVE_PATH = Path("data/meta_Home_and_Kitchen_furniture_siglip2_splits_20k.csv")
CATALOG_COLUMNS = [
    "asin", "description_original", "taxonomy_text", "furniture_subcategory",
    "category_path", "leaf_category", "image_path",
]


def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / PAIRS_RELATIVE_PATH).is_file():
            return candidate
    raise FileNotFoundError(f"Could not find {PAIRS_RELATIVE_PATH} from {start}")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
ARTIFACT_DIR = PROJECT_ROOT / "models/siglip2_furniture_text_strategy_benchmark"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(PROJECT_ROOT / PAIRS_RELATIVE_PATH, dtype="string").fillna("")
split_map = pd.read_csv(PROJECT_ROOT / SPLIT_RELATIVE_PATH, dtype="string").fillna("")
assert len(df) == len(split_map) == 20_000 and df["asin"].is_unique
df["leaf_category"] = df["category_path"].map(
    lambda value: [part.strip() for part in value.split(">") if part.strip()][-1]
)
df["image_file"] = df["image_path"].map(lambda value: PROJECT_ROOT / value)
assert df["image_file"].map(Path.is_file).all()
df = df.merge(split_map, on="asin", validate="one_to_one")
assert df["split"].value_counts().to_dict() == {"train": 16_000, "validation": 2_000, "test": 2_000}
train_df = df[df["split"].eq("train")].sample(n=TRAIN_ROWS, random_state=SEED).reset_index(drop=True)
eval_df = df[df["split"].eq("validation")].sample(n=EVAL_ROWS, random_state=SEED).reset_index(drop=True)
catalog_df = df[df["split"].eq("test")].sample(n=CATALOG_ROWS, random_state=SEED).reset_index(drop=True)
print({"train": len(train_df), "evaluation": len(eval_df), "catalog": len(catalog_df)})

{'train': 256, 'evaluation': 128, 'catalog': 500}


## 2. Construct compact and multi-chunk text variants

Compact text rewards corpus-distinguishing words and visible furniture attributes, penalizes boilerplate, and packs nonredundant sentences after a short leaf/subcategory prefix. Multi-chunk text keeps up to four evenly distributed overlapping chunks.

In [3]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
vectorizer = TfidfVectorizer(
    stop_words="english", lowercase=True, min_df=2, max_df=0.95, max_features=50_000
)
vectorizer.fit(df["description_original"])
idf_lookup = dict(zip(vectorizer.get_feature_names_out(), vectorizer.idf_))
DEFAULT_IDF = float(np.median(vectorizer.idf_))
WORD_RE = re.compile(r"[a-zA-Z][a-zA-Z0-9'-]*")
SENTENCE_RE = re.compile(r"(?<=[.!?])\s+|[\r\n]+|(?<=;)\s+")
VISUAL_TERMS = {
    "wood", "wooden", "walnut", "oak", "metal", "steel", "glass", "marble",
    "leather", "fabric", "velvet", "linen", "boucle", "rattan", "bamboo",
    "modern", "contemporary", "traditional", "rustic", "industrial", "farmhouse",
    "mid-century", "minimalist", "black", "white", "brown", "gray", "grey",
    "round", "oval", "rectangular", "square", "upholstered", "tufted", "storage",
    "drawer", "shelf", "adjustable", "folding", "swivel", "reclining", "tapered",
}
BOILERPLATE_TERMS = {
    "warranty", "shipping", "customer service", "contact us", "satisfaction guarantee",
    "package includes", "easy assembly", "assembly instructions",
}


def token_length(text, max_length=None):
    encoded = tokenizer(
        text, add_special_tokens=True, truncation=max_length is not None, max_length=max_length
    )
    return len(encoded["input_ids"])


def truncate_text(text, max_length):
    encoded = tokenizer(text, add_special_tokens=True, truncation=True, max_length=max_length)
    return tokenizer.decode(encoded["input_ids"], skip_special_tokens=True).strip()


def sentence_score(sentence):
    lowered = sentence.casefold()
    words = [word.casefold() for word in WORD_RE.findall(sentence)]
    if not words:
        return -1e9
    idfs = sorted((idf_lookup.get(word, DEFAULT_IDF) for word in set(words)), reverse=True)
    distinguishing = float(np.mean(idfs[: min(8, len(idfs))]))
    visual_bonus = 0.35 * sum(term in lowered for term in VISUAL_TERMS)
    numeric_bonus = 0.5 if re.search(r"\d+(?:\.\d+)?(?:[- ]?(?:inch|inches|cm|mm|ft|feet))", lowered) else 0.0
    boilerplate_penalty = 1.5 * sum(term in lowered for term in BOILERPLATE_TERMS)
    return distinguishing + visual_bonus + numeric_bonus - boilerplate_penalty


def compact_description(row, max_length=64):
    labels = [row.leaf_category]
    if row.furniture_subcategory not in labels and row.furniture_subcategory != "Unspecified":
        labels.append(row.furniture_subcategory)
    prefix = "; ".join(labels) + "."
    sentences = [part.strip() for part in SENTENCE_RE.split(row.description_original) if part.strip()]
    ranked = sorted(range(len(sentences)), key=lambda index: sentence_score(sentences[index]), reverse=True)
    selected_indices, selected_word_sets = [], []
    for index in ranked:
        words = {word.casefold() for word in WORD_RE.findall(sentences[index])}
        if any(len(words & prior) / max(1, len(words | prior)) > 0.65 for prior in selected_word_sets):
            continue
        candidate_indices = sorted(selected_indices + [index])
        candidate = prefix + " " + " ".join(sentences[i] for i in candidate_indices)
        if token_length(candidate) <= max_length:
            selected_indices.append(index)
            selected_word_sets.append(words)
    if not selected_indices and sentences:
        return truncate_text(prefix + " " + sentences[ranked[0]], max_length)
    result = prefix + (" " + " ".join(sentences[i] for i in sorted(selected_indices)) if selected_indices else "")
    return truncate_text(result, max_length)


def description_chunks(row, max_length=64, max_chunks=MAX_CHUNKS, overlap=CHUNK_OVERLAP):
    prefix = row.leaf_category + ". "
    prefix_tokens = tokenizer(prefix, add_special_tokens=False)["input_ids"]
    body_tokens = tokenizer(row.description_original, add_special_tokens=False)["input_ids"]
    body_budget = max(24, max_length - len(prefix_tokens) - 2)
    step = max(1, body_budget - overlap)
    windows = [body_tokens[start : start + body_budget] for start in range(0, len(body_tokens), step)] or [[]]
    if len(windows) > max_chunks:
        keep = np.linspace(0, len(windows) - 1, max_chunks).round().astype(int)
        windows = [windows[index] for index in sorted(set(keep))]
    return [
        truncate_text(prefix + tokenizer.decode(window, skip_special_tokens=True), max_length)
        for window in windows
    ]


def add_strategy_text(frame):
    frame = frame.copy()
    frame["description_compact"] = [compact_description(row) for row in frame.itertuples()]
    frame["description_chunks"] = [description_chunks(row) for row in frame.itertuples()]
    return frame


train_df = add_strategy_text(train_df)
eval_df = add_strategy_text(eval_df)
catalog_df = add_strategy_text(catalog_df)
assert train_df["description_compact"].map(token_length).le(64).all()
assert all(token_length(chunk) <= 64 for chunks in train_df["description_chunks"] for chunk in chunks)
print({
    "compact_median_tokens": float(train_df["description_compact"].map(token_length).median()),
    "chunks_per_product": train_df["description_chunks"].map(len).value_counts().sort_index().to_dict(),
})
display(train_df[["leaf_category", "description_original", "description_compact", "description_chunks"]].head())

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


{'compact_median_tokens': 59.0, 'chunks_per_product': {1: 43, 2: 60, 3: 43, 4: 110}}


,leaf_category,description_original,description_compact,description_chunks
0,Sofas & Couches,Divano Roma Furniture's Chesterfield Collectio...,Sofas & Couches; Living Room Furniture. Divano...,[Sofas & Couches. Divano Roma Furniture's Ches...
1,End Tables,Whether you're into minimalist design or looki...,End Tables; Living Room Furniture. The end tab...,[End Tables. Whether you're into minimalist de...
2,Furniture,"Whether you call it frugal or sensible, Prepar...",Furniture. Whether you call it frugal or sensi...,[Furniture. Whether you call it frugal or sens...
3,Home Office Desks,ErgoDesign Folding Computer Desk for Small Spa...,Home Office Desks; Home Office Furniture. Ergo...,[Home Office Desks. ErgoDesign Folding Compute...
4,Home Office Desks,The Ameriwood Home Wildwood Wood Veneer Audio ...,Home Office Desks; Home Office Furniture. The ...,[Home Office Desks. The Ameriwood Home Wildwoo...


## 3. Shared image/text encoding and training loaders

In [4]:
def open_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB")


class TrainingPairs(Dataset):
    def __init__(self, frame, texts):
        self.records = frame[["asin", "taxonomy_text", "image_file"]].to_dict("records")
        self.texts = list(texts)

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        return {
            "asin": row["asin"], "text": self.texts[index],
            "taxonomy_text": row["taxonomy_text"], "image": open_rgb(row["image_file"]),
        }


def make_training_loader(frame, texts, max_length):
    def collate(examples):
        batch = processor(
            text=[example["text"] for example in examples],
            images=[example["image"] for example in examples],
            padding="max_length", truncation=True, max_length=max_length, return_tensors="pt",
        )
        taxonomy = processor(
            text=[example["taxonomy_text"] for example in examples],
            padding="max_length", truncation=True, max_length=max_length, return_tensors="pt",
        )
        batch["taxonomy_input_ids"] = taxonomy["input_ids"]
        if "attention_mask" in taxonomy:
            batch["taxonomy_attention_mask"] = taxonomy["attention_mask"]
        return batch
    return DataLoader(
        TrainingPairs(frame, texts), batch_size=TRAIN_BATCH_SIZE, shuffle=True,
        generator=torch.Generator().manual_seed(SEED), num_workers=0,
        pin_memory=PIN_MEMORY, collate_fn=collate, drop_last=False,
    )


def image_batches(frame):
    for start in range(0, len(frame), ENCODE_BATCH_SIZE):
        images = [open_rgb(path) for path in frame.iloc[start : start + ENCODE_BATCH_SIZE]["image_file"]]
        yield processor(images=images, return_tensors="pt")


def encode_images(model, frame, description):
    chunks = []
    model.eval()
    with torch.inference_mode():
        for batch in tqdm(image_batches(frame), total=int(np.ceil(len(frame) / ENCODE_BATCH_SIZE)), desc=description):
            inputs = {key: value.to(DEVICE) for key, value in batch.items()}
            chunks.append(F.normalize(model.get_image_features(**inputs), dim=-1).cpu())
    return torch.cat(chunks)


def encode_texts(model, texts, max_length, description):
    chunks = []
    model.eval()
    with torch.inference_mode():
        for start in tqdm(range(0, len(texts), ENCODE_BATCH_SIZE), desc=description):
            batch = processor(
                text=list(texts[start : start + ENCODE_BATCH_SIZE]), padding="max_length",
                truncation=True, max_length=max_length, return_tensors="pt",
            )
            inputs = {key: value.to(DEVICE) for key, value in batch.items()}
            chunks.append(F.normalize(model.get_text_features(**inputs), dim=-1).cpu())
    return torch.cat(chunks)


def encode_multichunk_texts(model, chunk_lists, description):
    flat = [chunk for chunks in chunk_lists for chunk in chunks]
    owners = [owner for owner, chunks in enumerate(chunk_lists) for _ in chunks]
    flat_embeddings = encode_texts(model, flat, 64, description)
    sums = torch.zeros(len(chunk_lists), flat_embeddings.shape[1])
    counts = torch.zeros(len(chunk_lists), 1)
    for embedding, owner in zip(flat_embeddings, owners):
        sums[owner] += embedding
        counts[owner] += 1
    return F.normalize(sums / counts, dim=-1)

## 4. Model surgery, training, and metrics

In [5]:
def reset_seeds():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)


def expand_text_positions(model, new_length=128):
    embeddings = model.text_model.embeddings
    old_weight = embeddings.position_embedding.weight.detach()
    resized = F.interpolate(
        old_weight.T.unsqueeze(0), size=new_length, mode="linear", align_corners=True
    ).squeeze(0).T
    new_embedding = nn.Embedding(new_length, old_weight.shape[1]).to(old_weight.device)
    with torch.no_grad(): new_embedding.weight.copy_(resized)
    embeddings.position_embedding = new_embedding
    embeddings.register_buffer(
        "position_ids", torch.arange(new_length, device=old_weight.device).expand((1, -1)), persistent=False
    )
    model.config.text_config.max_position_embeddings = new_length


def load_model(strategy):
    reset_seeds()
    model = AutoModel.from_pretrained(MODEL_ID)
    if strategy == "extended128": expand_text_positions(model, 128)
    model = model.to(DEVICE)
    for parameter in model.parameters(): parameter.requires_grad = False
    for module in (model.text_model.head, model.vision_model.head):
        for parameter in module.parameters(): parameter.requires_grad = True
    model.logit_scale.requires_grad = True
    model.logit_bias.requires_grad = True
    if strategy == "extended128":
        model.text_model.embeddings.position_embedding.weight.requires_grad = True
    return model


def paired_recall(image_embeddings, text_embeddings, ks=(1, 5, 10)):
    similarities = image_embeddings @ text_embeddings.T
    targets = torch.arange(len(similarities))
    image_ranked = similarities.argsort(dim=1, descending=True)
    text_ranked = similarities.T.argsort(dim=1, descending=True)
    metrics = {}
    for k in ks:
        metrics[f"image_to_text_recall@{k}"] = (image_ranked[:, :k] == targets[:, None]).any(1).float().mean().item()
        metrics[f"text_to_image_recall@{k}"] = (text_ranked[:, :k] == targets[:, None]).any(1).float().mean().item()
    return metrics


def same_category_at_k(scores, labels, ks=(1, 5, 10), exclude_diagonal=True):
    scores = scores.clone()
    if exclude_diagonal and scores.shape[0] == scores.shape[1]: scores.fill_diagonal_(-torch.inf)
    ranked = scores.argsort(dim=1, descending=True).numpy()
    labels = np.asarray(labels)
    return {
        f"same_subcategory@{k}": float(np.mean([
            np.any(labels[ranked[index, :k]] == labels[index]) for index in range(len(labels))
        ]))
        for k in ks
    }


def category_query_precision(model, image_embeddings, text_embeddings, frame, max_length):
    counts = frame["leaf_category"].value_counts()
    leaves = counts[counts.ge(TOP_K)].head(QUERY_LEAF_COUNT).index.tolist()
    query_embeddings = encode_texts(model, leaves, max_length, "Category queries")
    labels = frame["leaf_category"].to_numpy()
    image_scores = query_embeddings @ image_embeddings.T
    text_scores = query_embeddings @ text_embeddings.T
    image_precisions, text_precisions = [], []
    for index, leaf in enumerate(leaves):
        image_top = image_scores[index].topk(TOP_K).indices.numpy()
        text_top = text_scores[index].topk(TOP_K).indices.numpy()
        image_precisions.append(float(np.mean(labels[image_top] == leaf)))
        text_precisions.append(float(np.mean(labels[text_top] == leaf)))
    return {
        "leaf_queries": leaves,
        "image_precision@10": float(np.mean(image_precisions)),
        "description_precision@10": float(np.mean(text_precisions)),
    }


def release_model(model):
    del model; gc.collect()
    if DEVICE.type == "mps": torch.mps.empty_cache()
    elif DEVICE.type == "cuda": torch.cuda.empty_cache()

In [6]:
def native_text_embeddings(model, strategy, frame):
    if strategy == "compact64":
        return encode_texts(model, frame["description_compact"].tolist(), 64, f"{strategy} native text")
    if strategy == "multichunk64":
        return encode_multichunk_texts(model, frame["description_chunks"].tolist(), f"{strategy} chunks")
    if strategy == "dual64":
        description = encode_texts(model, frame["description_original"].tolist(), 64, "dual description")
        taxonomy = encode_texts(model, frame["taxonomy_text"].tolist(), 64, "dual taxonomy")
        return F.normalize(
            (1 - DUAL_TAXONOMY_FUSION_WEIGHT) * description
            + DUAL_TAXONOMY_FUSION_WEIGHT * taxonomy, dim=-1,
        )
    max_length = 128 if strategy == "extended128" else 64
    return encode_texts(model, frame["description_original"].tolist(), max_length, f"{strategy} native text")


def training_texts(strategy, frame, epoch):
    if strategy == "compact64": return frame["description_compact"].tolist()
    if strategy == "multichunk64":
        return [chunks[epoch % len(chunks)] for chunks in frame["description_chunks"]]
    return frame["description_original"].tolist()


def run_strategy(strategy):
    max_length = 128 if strategy == "extended128" else 64
    model = load_model(strategy)
    trainable_names = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
    optimizer = torch.optim.AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    )
    use_amp = DEVICE.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    history, started = [], time.perf_counter()
    for epoch in range(EPOCHS):
        loader = make_training_loader(train_df, training_texts(strategy, train_df, epoch), max_length)
        model.train(); losses = []
        for batch in tqdm(loader, desc=f"{strategy} epoch {epoch + 1}"):
            optimizer.zero_grad(set_to_none=True)
            main_inputs = {
                key: value.to(DEVICE) for key, value in batch.items() if not key.startswith("taxonomy_")
            }
            autocast_context = torch.autocast("cuda", dtype=torch.float16) if use_amp else nullcontext()
            with autocast_context:
                main_loss = model(**main_inputs, return_loss=True, return_dict=True).loss
                if strategy == "dual64":
                    taxonomy_inputs = {
                        "pixel_values": main_inputs["pixel_values"],
                        "input_ids": batch["taxonomy_input_ids"].to(DEVICE),
                    }
                    if "taxonomy_attention_mask" in batch:
                        taxonomy_inputs["attention_mask"] = batch["taxonomy_attention_mask"].to(DEVICE)
                    taxonomy_loss = model(**taxonomy_inputs, return_loss=True, return_dict=True).loss
                    loss = (1 - DUAL_TAXONOMY_LOSS_WEIGHT) * main_loss + DUAL_TAXONOMY_LOSS_WEIGHT * taxonomy_loss
                else:
                    loss = main_loss
            if not torch.isfinite(loss): raise FloatingPointError(f"Non-finite {strategy} loss")
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [parameter for parameter in model.parameters() if parameter.requires_grad], MAX_GRAD_NORM
            )
            scaler.step(optimizer); scaler.update(); losses.append(loss.item())
        history.append({
            "epoch": epoch + 1, "mean_loss": float(np.mean(losses)),
            "first_loss": float(losses[0]), "last_loss": float(losses[-1]),
        })
    elapsed = time.perf_counter() - started

    eval_images = encode_images(model, eval_df, f"{strategy} eval images")
    eval_natural64 = encode_texts(
        model, eval_df["description_original"].tolist(), max_length, f"{strategy} natural text"
    )
    eval_native = native_text_embeddings(model, strategy, eval_df)
    catalog_images = encode_images(model, catalog_df, f"{strategy} catalog images")
    catalog_native = native_text_embeddings(model, strategy, catalog_df)
    query_metrics = category_query_precision(model, catalog_images, catalog_native, catalog_df, max_length)
    natural_metrics = paired_recall(eval_images, eval_natural64)
    native_metrics = paired_recall(eval_images, eval_native)
    labels = catalog_df["furniture_subcategory"].to_numpy()
    image_category = same_category_at_k(catalog_images @ catalog_images.T, labels)
    cross_category = same_category_at_k(catalog_images @ catalog_native.T, labels)
    fixed_query = encode_texts(model, [TEXT_QUERY], max_length, f"{strategy} fixed query")[0]
    fixed_scores = catalog_images @ fixed_query
    values, indices = fixed_scores.topk(TOP_K)
    fixed_results = catalog_df.iloc[indices.tolist()][CATALOG_COLUMNS].copy()
    fixed_results.insert(1, "score", values.numpy())

    checkpoint = ARTIFACT_DIR / f"{strategy}_heads.pt"
    torch.save(
        {name: parameter.detach().cpu() for name, parameter in model.named_parameters() if name in trainable_names},
        checkpoint,
    )
    np.save(ARTIFACT_DIR / f"{strategy}_image_embeddings.npy", catalog_images.numpy())
    np.save(ARTIFACT_DIR / f"{strategy}_text_embeddings.npy", catalog_native.numpy())
    fixed_results.to_csv(ARTIFACT_DIR / f"{strategy}_fixed_query_results.csv", index=False)
    result = {
        "strategy": strategy, "max_length": max_length,
        "trainable_parameters": sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad),
        "elapsed_seconds": elapsed, "history": history,
        "natural_metrics": natural_metrics, "native_metrics": native_metrics,
        "category_query_metrics": query_metrics,
        "image_category_metrics": image_category,
        "cross_category_metrics": cross_category,
        "fixed_query_asins": fixed_results["asin"].tolist(),
    }
    print({"strategy": strategy, "elapsed_seconds": round(elapsed, 1), "history": history})
    release_model(model)
    return result

## 5. Run five fresh models sequentially

In [7]:
baseline64 = run_strategy("baseline64")

baseline64 epoch 1:   0%|          | 0/64 [00:00<?, ?it/s]

baseline64 epoch 2:   0%|          | 0/64 [00:00<?, ?it/s]

baseline64 eval images:   0%|          | 0/16 [00:00<?, ?it/s]

baseline64 natural text:   0%|          | 0/16 [00:00<?, ?it/s]

baseline64 native text:   0%|          | 0/16 [00:00<?, ?it/s]

baseline64 catalog images:   0%|          | 0/63 [00:00<?, ?it/s]

baseline64 native text:   0%|          | 0/63 [00:00<?, ?it/s]

Category queries:   0%|          | 0/3 [00:00<?, ?it/s]

baseline64 fixed query:   0%|          | 0/1 [00:00<?, ?it/s]

{'strategy': 'baseline64', 'elapsed_seconds': 50.8, 'history': [{'epoch': 1, 'mean_loss': 1.8661522765178233, 'first_loss': 1.7432979345321655, 'last_loss': 2.508241653442383}, {'epoch': 2, 'mean_loss': 0.8162970463745296, 'first_loss': 0.12177704274654388, 'last_loss': 1.7174394130706787}]}


In [8]:
compact64 = run_strategy("compact64")

compact64 epoch 1:   0%|          | 0/64 [00:00<?, ?it/s]

compact64 epoch 2:   0%|          | 0/64 [00:00<?, ?it/s]

compact64 eval images:   0%|          | 0/16 [00:00<?, ?it/s]

compact64 natural text:   0%|          | 0/16 [00:00<?, ?it/s]

compact64 native text:   0%|          | 0/16 [00:00<?, ?it/s]

compact64 catalog images:   0%|          | 0/63 [00:00<?, ?it/s]

compact64 native text:   0%|          | 0/63 [00:00<?, ?it/s]

Category queries:   0%|          | 0/3 [00:00<?, ?it/s]

compact64 fixed query:   0%|          | 0/1 [00:00<?, ?it/s]

{'strategy': 'compact64', 'elapsed_seconds': 36.4, 'history': [{'epoch': 1, 'mean_loss': 1.9712682932149619, 'first_loss': 4.064370155334473, 'last_loss': 0.6862279772758484}, {'epoch': 2, 'mean_loss': 0.8428692963498179, 'first_loss': 1.0242711305618286, 'last_loss': 0.3048000931739807}]}


In [9]:
multichunk64 = run_strategy("multichunk64")

multichunk64 epoch 1:   0%|          | 0/64 [00:00<?, ?it/s]

multichunk64 epoch 2:   0%|          | 0/64 [00:00<?, ?it/s]

multichunk64 eval images:   0%|          | 0/16 [00:00<?, ?it/s]

multichunk64 natural text:   0%|          | 0/16 [00:00<?, ?it/s]

multichunk64 chunks:   0%|          | 0/44 [00:00<?, ?it/s]

multichunk64 catalog images:   0%|          | 0/63 [00:00<?, ?it/s]

multichunk64 chunks:   0%|          | 0/176 [00:00<?, ?it/s]

Category queries:   0%|          | 0/3 [00:00<?, ?it/s]

multichunk64 fixed query:   0%|          | 0/1 [00:00<?, ?it/s]

{'strategy': 'multichunk64', 'elapsed_seconds': 38.7, 'history': [{'epoch': 1, 'mean_loss': 1.9148545359494165, 'first_loss': 3.18813157081604, 'last_loss': 2.457376480102539}, {'epoch': 2, 'mean_loss': 1.5624526853207499, 'first_loss': 0.7589005827903748, 'last_loss': 0.6982444524765015}]}


In [10]:
dual64 = run_strategy("dual64")

dual64 epoch 1:   0%|          | 0/64 [00:00<?, ?it/s]

dual64 epoch 2:   0%|          | 0/64 [00:00<?, ?it/s]

dual64 eval images:   0%|          | 0/16 [00:00<?, ?it/s]

dual64 natural text:   0%|          | 0/16 [00:00<?, ?it/s]

dual description:   0%|          | 0/16 [00:00<?, ?it/s]

...

dual description:   0%|          | 0/63 [00:00<?, ?it/s]

dual taxonomy:   0%|          | 0/63 [00:00<?, ?it/s]

Category queries:   0%|          | 0/3 [00:00<?, ?it/s]

dual64 fixed query:   0%|          | 0/1 [00:00<?, ?it/s]

{'strategy': 'dual64', 'elapsed_seconds': 61.3, 'history': [{'epoch': 1, 'mean_loss': 2.624946366995573, 'first_loss': 4.968003273010254, 'last_loss': 2.3602027893066406}, {'epoch': 2, 'mean_loss': 1.2482395973056555, 'first_loss': 0.8270618915557861, 'last_loss': 1.7775322198867798}]}


In [11]:
extended128 = run_strategy("extended128")

extended128 epoch 1:   0%|          | 0/64 [00:00<?, ?it/s]

extended128 epoch 2:   0%|          | 0/64 [00:00<?, ?it/s]

extended128 eval images:   0%|          | 0/16 [00:00<?, ?it/s]

extended128 natural text:   0%|          | 0/16 [00:00<?, ?it/s]

extended128 native text:   0%|          | 0/16 [00:00<?, ?it/s]

extended128 catalog images:   0%|          | 0/63 [00:00<?, ?it/s]

extended128 native text:   0%|          | 0/63 [00:00<?, ?it/s]

Category queries:   0%|          | 0/3 [00:00<?, ?it/s]

extended128 fixed query:   0%|          | 0/1 [00:00<?, ?it/s]

{'strategy': 'extended128', 'elapsed_seconds': 50.3, 'history': [{'epoch': 1, 'mean_loss': 2.234680073102936, 'first_loss': 3.5207431316375732, 'last_loss': 0.9587422609329224}, {'epoch': 2, 'mean_loss': 0.9341423282166943, 'first_loss': 0.6321471333503723, 'last_loss': 0.6918476223945618}]}


## 6. Compare strategies

Natural metrics use original descriptions. Native metrics use each strategy's operational representation. Category queries are the 20 most common leaf types with at least ten catalog examples.

In [12]:
results = [baseline64, compact64, multichunk64, dual64, extended128]
rows = []
for result in results:
    row = {
        "strategy": result["strategy"],
        "max_length": result["max_length"],
        "trainable_parameters": result["trainable_parameters"],
        "training_seconds": result["elapsed_seconds"],
        "final_mean_loss": result["history"][-1]["mean_loss"],
        "query_image_precision@10": result["category_query_metrics"]["image_precision@10"],
        "query_description_precision@10": result["category_query_metrics"]["description_precision@10"],
        "image_neighbor_category@1": result["image_category_metrics"]["same_subcategory@1"],
        "cross_neighbor_category@1": result["cross_category_metrics"]["same_subcategory@1"],
    }
    row.update({f"natural_{key}": value for key, value in result["natural_metrics"].items()})
    row.update({f"native_{key}": value for key, value in result["native_metrics"].items()})
    rows.append(row)
summary_df = pd.DataFrame(rows).set_index("strategy")
display(summary_df.T)

query_overlap = pd.DataFrame(
    {result["strategy"]: result["fixed_query_asins"] for result in results}
)
display(query_overlap)
summary_df.to_csv(ARTIFACT_DIR / "strategy_summary.csv")
query_overlap.to_csv(ARTIFACT_DIR / "fixed_query_rankings.csv", index=False)
catalog_df[CATALOG_COLUMNS].to_csv(ARTIFACT_DIR / "embedding_catalog.csv", index=False)
with (ARTIFACT_DIR / "experiment.json").open("w") as output:
    json.dump({
        "model_id": MODEL_ID, "seed": SEED, "train_rows": TRAIN_ROWS,
        "eval_rows": EVAL_ROWS, "catalog_rows": CATALOG_ROWS, "epochs": EPOCHS,
        "text_query": TEXT_QUERY, "results": results,
    }, output, indent=2)
print("Saved artifacts:", sorted(path.name for path in ARTIFACT_DIR.iterdir()))

strategy,baseline64,compact64,multichunk64,dual64,extended128
max_length,6.400000e+01,6.400000e+01,6.400000e+01,6.400000e+01,1.280000e+02
trainable_parameters,7.677698e+06,7.677698e+06,7.677698e+06,7.677698e+06,7.776002e+06
training_seconds,5.081298e+01,3.635072e+01,3.873290e+01,6.130726e+01,5.031410e+01
final_mean_loss,8.162970e-01,8.428693e-01,1.562453e+00,1.248240e+00,9.341423e-01
query_image_precision@10,1.333333e-01,1.500000e-01,1.611111e-01,1.388889e-01,7.222222e-02
query_description_precision@10,6.111111e-02,1.444444e-01,1.500000e-01,1.277778e-01,3.333333e-02
image_neighbor_category@1,7.000000e-01,6.920000e-01,6.940000e-01,6.960000e-01,6.980000e-01
cross_neighbor_category@1,6.420000e-01,6.440000e-01,6.800000e-01,6.480000e-01,6.020000e-01
natural_image_to_text_recall@1,4.687500e-01,4.687500e-01,4.609375e-01,4.453125e-01,3.984375e-01
natural_text_to_image_recall@1,4.375000e-01,4.609375e-01,4.453125e-01,4.531250e-01,3.671875e-01


,baseline64,compact64,multichunk64,dual64,extended128
0,B08SSDJ8TC,B08SSDJ8TC,B08SSDJ8TC,B08SSDJ8TC,B003AVLBOU
1,B075LNVVNQ,B075LNVVNQ,B075LNVVNQ,B075LNVVNQ,B08SSDJ8TC
2,B01JGY2ECM,B01JGY2ECM,B01JGY2ECM,B01JGY2ECM,B01JGY2ECM
3,B003AVLBOU,B003AVLBOU,B003AVLBOU,B007U3LBGW,B01FXYB86M
4,B007U3LBGW,B007U3LBGW,B007U3LBGW,B003AVLBOU,B01MZ4G2HJ
5,B07XB1P8Q6,B07XB1P8Q6,B07XB1P8Q6,B07XB1P8Q6,B07XB1P8Q6
6,B01FXYB86M,B01FXYB86M,B01FXYB86M,B08X2D2685,B07B6DDJCW
7,B08X2D2685,B08X2D2685,B08X2D2685,B01FXYB86M,B007U3LBGW
8,B078K7MWRR,B01MZ4G2HJ,B078K7MWRR,B078K7MWRR,B00KALWPIO
9,B01MZ4G2HJ,B078K7MWRR,B01MZ4G2HJ,B01MZ4G2HJ,B004DSGMO8


Saved artifacts: ['baseline64_fixed_query_results.csv', 'baseline64_heads.pt', 'baseline64_image_embeddings.npy', 'baseline64_text_embeddings.npy', 'compact64_fixed_query_results.csv', 'compact64_heads.pt', 'compact64_image_embeddings.npy', 'compact64_text_embeddings.npy', 'dual64_fixed_query_results.csv', 'dual64_heads.pt', 'dual64_image_embeddings.npy', 'dual64_text_embeddings.npy', 'embedding_catalog.csv', 'experiment.json', 'extended128_fixed_query_results.csv', 'extended128_heads.pt', 'extended128_image_embeddings.npy', 'extended128_text_embeddings.npy', 'fixed_query_rankings.csv', 'multichunk64_fixed_query_results.csv', 'multichunk64_heads.pt', 'multichunk64_image_embeddings.npy', 'multichunk64_text_embeddings.npy', 'strategy_summary.csv']


## Selection guardrails

- Prefer strategies that improve multiple retrieval views, not only training loss.
- Treat the 128-token arm cautiously: it changes the architecture and trains new positional parameters from limited data.
- A screening winner should be confirmed on the full 2,000-product catalog and multiple seeds before replacing the current pipeline.
- Inspect fixed-query rankings for semantic quality and not only category labels.